In [0]:
from pyspark.sql.functions import current_timestamp, lit

raw_path = "/Volumes/workspace/default/football_database/football_database"

tables = ["appearances", "games", "leagues", "players", "shots", "teams", "teamstats"]

for t in tables:
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(f"{raw_path}/{t}.csv"))
    
    df = (df
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", lit(f"{t}.csv")))
    
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"bronze_{t}"))

In [0]:
for t in tables:
    df = spark.table(f"bronze_{t}")
    print(f"{t}: {df.count()} riviä, {len(df.columns)} saraketta")

In [0]:
%sql
SHOW TABLES IN workspace.default

In [0]:
from pyspark.sql.functions import col, count, when, isnan

tables = ["appearances", "games", "leagues", "players", "shots", "teams", "teamstats"]

def profile_table(table_name):
    df = spark.table(f"bronze_{table_name}")
    total_rows = df.count()
    print(f"\n=== {table_name} ({total_rows} riviä) ===")
    
    for c, dtype in df.dtypes:
        if c.startswith("_"):  # ohita metadata-sarakkeet
            continue
        null_count = df.filter(col(c).isNull()).count()
        distinct_count = df.select(c).distinct().count()
        null_pct = round(100 * null_count / total_rows, 1) if total_rows > 0 else 0
        print(f"  {c:<30} {dtype:<15} nullit: {null_count} ({null_pct}%)  eri arvoja: {distinct_count}")

for t in tables:
    profile_table(t)

In [0]:
from pyspark.sql.functions import col, expr, lit

odds_cols = ["B365H","B365D","B365A","BWH","BWD","BWA","IWH","IWD","IWA",
             "PSH","PSD","PSA","WHH","WHD","WHA","VCH","VCD","VCA",
             "PSCH","PSCD","PSCA"]

games = spark.table("bronze_games")
results = None

for c in odds_cols:
    r = (games
         .filter(col(c).isNotNull() & expr(f"try_cast(`{c}` AS DOUBLE)").isNull())
         .groupBy(col(c).alias("arvo"))
         .count()
         .withColumn("sarake", lit(c)))
    results = r if results is None else results.unionByName(r)

results.select("sarake", "arvo", "count").orderBy("sarake", "arvo").show(200, truncate=False)

In [0]:
spark.sql("""
    SELECT yellowCards, COUNT(*) AS n
    FROM bronze_teamstats
    WHERE TRY_CAST(yellowCards AS INT) IS NULL
    GROUP BY yellowCards
""").show(50, truncate=False)

spark.sql("""
    SELECT assisterID, COUNT(*) AS n
    FROM bronze_shots
    WHERE TRY_CAST(assisterID AS INT) IS NULL
    GROUP BY assisterID
""").show(50, truncate=False)

In [0]:
spark.sql("""
    SELECT gameID, teamID, date, location, goals, redCards, yellowCards
    FROM bronze_teamstats
    WHERE yellowCards = 'NA'
""").show(truncate=False)

In [0]:
spark.sql("""
    SELECT situation,
           COUNT(*) AS shots,
           SUM(CASE WHEN assisterID = 'NA' THEN 1 ELSE 0 END) AS no_assister,
           ROUND(100 * AVG(CASE WHEN assisterID = 'NA' THEN 1.0 ELSE 0.0 END), 1) AS percentage
    FROM bronze_shots
    GROUP BY situation
    ORDER BY shots DESC
""").show(truncate=False)